# ⚖️ Model Selection & Tradeoffs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlwhyte/GenAI_foundry/blob/main/notebooks/model_selection_tradeoffs.ipynb)

**No API key required** · Part 1 of the GenAI Foundry series

## Learning Goals
By the end of this notebook you will be able to:
- Understand the key dimensions for comparing LLMs: capability, speed, cost, and context
- Use a structured framework to choose the right model for a task
- Calculate real-world token costs for common use cases
- Explain the tradeoffs between open-source and proprietary models

> 📚 **Prerequisites:** Complete *Tokens & Embeddings* first — this notebook builds on your understanding of tokens and context windows.

---
## Part 1: The LLM Landscape

In 2025, there are hundreds of available LLMs. How do you choose?

Models differ across four key dimensions:

| Dimension | What it means | Example tradeoff |
|---|---|---|
| **Capability** | How well it reasons, writes, codes | GPT-4o > GPT-3.5 for complex tasks |
| **Speed** | Tokens generated per second | Smaller models are faster |
| **Cost** | Price per 1M tokens | Can vary 100x between models |
| **Context window** | Max tokens in one call | 4K vs 128K vs 1M tokens |

There's no single 'best' model — the right choice depends on your task.

In [ ]:
# Model catalog (approximate as of 2025 — prices change frequently!)
# Sources: official provider pricing pages

MODELS = [
    # name, provider, context_k, input_per_1M, output_per_1M, speed_relative, tier
    {'name': 'GPT-4o',              'provider': 'OpenAI',    'context_k': 128,  'input': 2.50,  'output': 10.00, 'speed': 'medium', 'tier': 'frontier'},
    {'name': 'GPT-4o mini',         'provider': 'OpenAI',    'context_k': 128,  'input': 0.15,  'output': 0.60,  'speed': 'fast',   'tier': 'efficient'},
    {'name': 'Claude 3.5 Sonnet',   'provider': 'Anthropic', 'context_k': 200,  'input': 3.00,  'output': 15.00, 'speed': 'medium', 'tier': 'frontier'},
    {'name': 'Claude 3 Haiku',      'provider': 'Anthropic', 'context_k': 200,  'input': 0.25,  'output': 1.25,  'speed': 'fast',   'tier': 'efficient'},
    {'name': 'Gemini 1.5 Pro',      'provider': 'Google',    'context_k': 1000, 'input': 1.25,  'output': 5.00,  'speed': 'medium', 'tier': 'frontier'},
    {'name': 'Gemini 1.5 Flash',    'provider': 'Google',    'context_k': 1000, 'input': 0.075, 'output': 0.30,  'speed': 'fast',   'tier': 'efficient'},
    {'name': 'Llama 3.1 70B',       'provider': 'Meta/OSS',  'context_k': 128,  'input': 0.59,  'output': 0.79,  'speed': 'medium', 'tier': 'open-source'},
    {'name': 'Llama 3.2 3B',        'provider': 'Meta/OSS',  'context_k': 128,  'input': 0.06,  'output': 0.06,  'speed': 'fast',   'tier': 'open-source'},
    {'name': 'Mistral 7B',          'provider': 'Mistral/OSS','context_k': 32,  'input': 0.25,  'output': 0.25,  'speed': 'fast',   'tier': 'open-source'},
    {'name': 'Phi-3 Mini',          'provider': 'Microsoft',  'context_k': 128,  'input': 0.13,  'output': 0.13,  'speed': 'fast',   'tier': 'small'},
]

print(f"{'Model':<25} {'Provider':<12} {'Context':<10} {'$/1M in':<10} {'$/1M out':<10} {'Tier'}")
print('-' * 80)
for m in MODELS:
    print(f"{m['name']:<25} {m['provider']:<12} {m['context_k']:>5}K     "
          f"${m['input']:<9.3f} ${m['output']:<9.3f} {m['tier']}")

---
## Part 2: Real-World Cost Calculator

Pricing is per 1 million tokens. Let's make this concrete with real use cases.

In [ ]:
def calculate_cost(model, input_tokens, output_tokens, num_calls=1):
    """Calculate the cost of an LLM call."""
    input_cost  = (input_tokens  / 1_000_000) * model['input']
    output_cost = (output_tokens / 1_000_000) * model['output']
    total_per_call = input_cost + output_cost
    return total_per_call * num_calls

# Common use cases
USE_CASES = [
    {
        'name': 'Simple Q&A chatbot message',
        'input_tokens': 200,    # system prompt + short user message
        'output_tokens': 150,   # short answer
        'calls_per_day': 1000,
    },
    {
        'name': 'Document summarisation (10 pages)',
        'input_tokens': 8000,   # 10 pages ≈ 8K tokens
        'output_tokens': 500,   # summary
        'calls_per_day': 100,
    },
    {
        'name': 'RAG pipeline (retrieve + answer)',
        'input_tokens': 3000,   # system + 3 retrieved chunks + question
        'output_tokens': 300,   # grounded answer
        'calls_per_day': 500,
    },
    {
        'name': 'Code generation (complex function)',
        'input_tokens': 1000,   # context + instructions
        'output_tokens': 800,   # generated code
        'calls_per_day': 200,
    },
]

print('=== DAILY COST BY USE CASE AND MODEL ===\n')
for uc in USE_CASES:
    print(f"Use case: {uc['name']}")
    print(f"  Tokens: {uc['input_tokens']:,} in / {uc['output_tokens']:,} out × {uc['calls_per_day']:,} calls/day")

    costs = []
    for m in MODELS:
        daily = calculate_cost(m, uc['input_tokens'], uc['output_tokens'], uc['calls_per_day'])
        costs.append((m['name'], daily))
    costs.sort(key=lambda x: x[1])

    print(f"  {'Model':<25} Daily Cost    Monthly Est.")
    for name, daily in costs:
        monthly = daily * 30
        bar = '█' * min(int(daily * 200), 30)
        print(f"  {name:<25} ${daily:>7.4f}      ${monthly:>8.2f}   {bar}")
    print()

---
## Part 3: Context Window — Why It Matters

The context window is the maximum amount of text the model can 'see' at once — including your prompt, history, and the response.

When you hit the limit, the model can't see earlier parts of the conversation.

In [ ]:
# Approximate token counts for common content types
CONTENT_SIZES = {
    'Short tweet / message':         50,
    'Email (1 page)':               500,
    'Short article (5 pages)':     4_000,
    'Research paper (20 pages)':  16_000,
    'Short book (100 pages)':     80_000,
    'Novel (300 pages)':         240_000,
    '1 hour of meeting transcript': 12_000,
    'Entire codebase (small)':    50_000,
}

print('=== WHAT FITS IN EACH MODEL\'S CONTEXT WINDOW? ===\n')
for m in MODELS:
    context_tokens = m['context_k'] * 1000
    fits = [name for name, tokens in CONTENT_SIZES.items() if tokens <= context_tokens]
    print(f"{m['name']:<25} ({m['context_k']}K tokens)")
    for item in fits:
        print(f'  ✅ {item}')
    print()

print('Key insight: larger context = more flexibility but higher cost per call.')
print('Gemini 1.5 Pro\'s 1M context can fit entire codebases or books!')

---
## Part 4: A Decision Framework

Use this framework to choose the right model for any task:

In [ ]:
def recommend_model(task_description, budget='any', needs_long_context=False, 
                    data_privacy=False, latency='any'):
    """
    A simple model recommendation framework.
    
    In practice, run benchmarks on your actual task — this is a starting point.
    """
    recommendations = []
    reasons = []

    if data_privacy:
        recommendations.append('Llama 3.1 70B (self-hosted)')
        recommendations.append('Mistral 7B (self-hosted)')
        reasons.append('🔒 Data privacy: open-source models run locally, no data leaves your infrastructure')
        print('\n'.join(['📋 RECOMMENDATION:'] + [f'  → {r}' for r in recommendations]))
        print('\nReasons:')
        for r in reasons: print(f'  {r}')
        return

    if needs_long_context:
        recommendations.append('Gemini 1.5 Pro (1M context)')
        recommendations.append('Gemini 1.5 Flash (1M context, cheaper)')
        recommendations.append('Claude 3.5 Sonnet (200K context)')
        reasons.append('📄 Long context required: only Gemini and Claude offer >128K')

    if budget == 'tight':
        recommendations = ['GPT-4o mini', 'Gemini 1.5 Flash', 'Claude 3 Haiku', 'Llama 3.2 3B']
        reasons.append('💰 Budget constraint: efficient-tier models are 10-50x cheaper than frontier')
    elif budget == 'unlimited':
        recommendations = ['GPT-4o', 'Claude 3.5 Sonnet', 'Gemini 1.5 Pro']
        reasons.append('🏆 Best capability: frontier models for complex reasoning tasks')

    if latency == 'low':
        recommendations = [m for m in recommendations if any(
            x in m for x in ['mini', 'Flash', 'Haiku', '3B', '7B']
        )] or ['GPT-4o mini', 'Gemini 1.5 Flash']
        reasons.append('⚡ Low latency: prefer smaller/efficient models for real-time use')

    if not recommendations:
        recommendations = ['GPT-4o mini', 'Claude 3 Haiku']
        reasons.append('📊 General use: efficient models cover most tasks at low cost')

    print('📋 RECOMMENDATIONS:')
    for r in recommendations[:3]:
        print(f'  → {r}')
    print('\nReasons:')
    for r in reasons:
        print(f'  {r}')
    print('\n⚠️  Always benchmark on your specific task before committing to a model!')


# Try different scenarios
print('=' * 60)
print('Scenario 1: Customer support chatbot, cost-sensitive')
print('=' * 60)
recommend_model('customer support', budget='tight', latency='low')

print()
print('=' * 60)
print('Scenario 2: Analyse 200-page legal contracts')
print('=' * 60)
recommend_model('legal document analysis', needs_long_context=True)

print()
print('=' * 60)
print('Scenario 3: Internal HR tool, strict data privacy')
print('=' * 60)
recommend_model('HR data processing', data_privacy=True)

---
## Part 5: Open-Source vs. Proprietary

One of the biggest model decisions is whether to use an open-source model you run yourself, or a proprietary API.

In [ ]:
comparison = {
    'Factor': [
        'Data privacy',
        'Cost at scale',
        'Setup complexity',
        'Maintenance burden',
        'Cutting-edge capability',
        'Customisation (fine-tuning)',
        'Reliability / uptime',
        'Support',
    ],
    'Open-Source (self-hosted)': [
        '✅ Data never leaves your infra',
        '✅ Fixed compute cost, no per-token fees',
        '❌ Requires GPU infra / Docker setup',
        '❌ You manage updates and scaling',
        '🟡 6-12 months behind frontier',
        '✅ Full access to weights',
        '🟡 Depends on your infra',
        '🟡 Community / forums',
    ],
    'Proprietary API': [
        '❌ Data sent to provider',
        '🟡 Pay per token, expensive at scale',
        '✅ One API call — no infra needed',
        '✅ Provider handles everything',
        '✅ State-of-the-art',
        '❌ No weight access',
        '✅ Enterprise SLAs available',
        '✅ Commercial support',
    ]
}

print(f"{'Factor':<35} {'Open-Source':<35} {'Proprietary API'}")
print('-' * 100)
for i, factor in enumerate(comparison['Factor']):
    oss = comparison['Open-Source (self-hosted)'][i]
    prop = comparison['Proprietary API'][i]
    print(f"{factor:<35} {oss:<35} {prop}")

print()
print('Rule of thumb:')
print('  → Start with a proprietary API (GPT-4o mini or equivalent)')
print('  → Switch to open-source when: privacy required, volume is high, or you need fine-tuning')

---
## Part 6: Benchmarks — Capability Comparison

Benchmarks measure model capability on standardised tasks. Use them as a starting point, then run your own evals on tasks that match your use case.

In [ ]:
# Approximate benchmark scores (as of early 2025)
# MMLU: general knowledge, HumanEval: coding, MATH: mathematics
# Scores are % correct — higher is better

benchmarks = [
    # name,                MMLU,  HumanEval, MATH,  note
    ('GPT-4o',             88.7,  90.2,      76.6,  'Frontier'),
    ('Claude 3.5 Sonnet',  88.3,  92.0,      71.1,  'Frontier'),
    ('Gemini 1.5 Pro',     85.9,  84.1,      67.7,  'Frontier'),
    ('GPT-4o mini',        82.0,  87.2,      70.2,  'Efficient'),
    ('Claude 3 Haiku',     75.2,  75.9,      38.9,  'Efficient'),
    ('Gemini 1.5 Flash',   78.9,  74.3,      54.9,  'Efficient'),
    ('Llama 3.1 70B',      83.6,  80.5,      65.7,  'Open-Source'),
    ('Mistral 7B',         64.2,  37.8,      22.4,  'Open-Source Small'),
    ('Phi-3 Mini',         68.8,  58.1,      33.5,  'Open-Source Small'),
]

print(f"{'Model':<25} {'MMLU':<8} {'HumanEval':<12} {'MATH':<8} {'Tier'}")
print('-' * 70)
for name, mmlu, he, math, note in benchmarks:
    bar_mmlu = '█' * int(mmlu / 10)
    print(f"{name:<25} {mmlu:>5.1f}%  {he:>8.1f}%   {math:>5.1f}%   {note}")

print()
print('Key insights:')
print('  1. Frontier models score 85-92% on most benchmarks')
print('  2. Efficient models (mini/flash/haiku) score 75-88% at 10-50x lower cost')
print('  3. Small open-source models (7B) score 60-70% — good for simple tasks')
print('  4. Benchmark ≠ performance on YOUR task — always run your own evaluation!')

---
## Part 7: Quick-Reference Cheat Sheet

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           MODEL SELECTION CHEAT SHEET (2025)                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  START HERE → GPT-4o mini / Gemini Flash / Claude Haiku     ║
║  Fast, cheap, surprisingly capable for most tasks           ║
║                                                              ║
║  UPGRADE TO frontier if:                                     ║
║  • Complex multi-step reasoning                             ║
║  • High-stakes decisions (legal, medical)                   ║
║  • Advanced coding or math                                  ║
║                                                              ║
║  USE LONG-CONTEXT (Gemini 1.5 Pro) if:                       ║
║  • Processing documents > 50 pages                         ║
║  • Entire codebase analysis                                 ║
║  • Multi-hour meeting transcripts                           ║
║                                                              ║
║  GO OPEN-SOURCE (Llama, Mistral) if:                         ║
║  • Data privacy is non-negotiable                           ║
║  • Volume is very high (millions of calls/day)              ║
║  • You need fine-tuning on custom data                      ║
║                                                              ║
║  COST RULES OF THUMB:                                        ║
║  • Efficient tier: ~$0.001 per 1K tokens                    ║
║  • Frontier tier: ~$0.010 per 1K tokens (10x more)         ║
║  • Self-hosted GPU: fixed cost, ~$1-3/hour per GPU          ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## 📝 Summary

You've learned the four dimensions of model comparison: **capability, speed, cost, and context window**. Here are the key takeaways:

- There is no universally 'best' model — the right choice depends on your task, budget, privacy needs, and latency requirements
- Start with an efficient model (GPT-4o mini, Gemini Flash, Claude Haiku) — they handle ~80% of tasks at 10-50x lower cost
- Open-source models (Llama, Mistral) are compelling for privacy, high volume, or fine-tuning
- Always run your own benchmarks on your specific task — published benchmarks are a starting point, not a verdict
- Costs can vary 100x between models, so model selection is one of the highest-leverage decisions in any AI product

## 🚀 What's Next?

- **Notebook 5: Simple Chatbot** — build your first LLM-powered chatbot (requires OpenAI API key)
- **AgenticAI Foundry → Module 1: LLM Cost Explorer** — an interactive tool to compare model costs across real use cases
- Keep an eye on [lmsys.org/chatbot-arena](https://lmsys.org/chatbot-arena/) for live human-preference rankings